# SemiTNet — Reproduction & Evaluation Notebook

این Notebook گزارش پژوهشگرمحور و نقطه‌ی ورود واحد مخزن است. سه مسیر علمی را جدا نگه می‌دارد:

1. **Reduced measured simulation:** اجرای واقعی `Teacher → Pseudo labels → Student → EMA → Evaluation` با `QuickSemiTransformer`.
2. **Strict paper-equivalence:** gateهای `G0–G3`؛ بدون عبور شواهد، ادعای reproduction ممنوع است.
3. **TED3 defensible reimplementation:** فقط با `data/train/`، `data/test/` و `data/TED3-unlabeled-data-15k-pseudo-mask/`.

`Input X-ray → Encoder → Segmentation/Query Prediction → Teacher/Student → Pseudo Labels → EMA/Distillation → IoU/Dice/Precision/Recall/F1`

> هیچ package، download، extraction یا full 26,250-iteration training به‌صورت خودکار انجام نمی‌شود.

In [ ]:
# هدف: تعیین root، کنترل اجرای امن و اعتبارسنجی environment.
# ورودی: current working directory و Python environment.
# خروجی: ROOT، versions، CUDA/GPU، dependency status و Git HEAD.
# رفتار مورد انتظار: هیچ package نصب نمی‌شود و training سنگین خاموش است.
from pathlib import Path
import os,sys,json,subprocess,hashlib,importlib.util,platform,shlex
ROOT=next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'project.py').is_file() and (p/'reproduction/reference_contract.json').is_file()),None)
if ROOT is None: raise RuntimeError('root مخزن salehi-semiTNet پیدا نشد')
os.chdir(ROOT)
RUN_SAFE_COMMANDS=RUN_REDUCED_SIMULATION=RUN_FULL_HASH_AUDIT=RUN_TED3_EXPENSIVE_TRAINING=False
print('ROOT:',ROOT); print('Python:',sys.version.replace('\n',' ')); print('Platform:',platform.platform())
for n in ['torch','numpy','pandas','matplotlib','PIL','yaml','detectron2']: print(n,'available=',importlib.util.find_spec(n) is not None)
try:
 import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except Exception as e: print('PyTorch:',e)
g=subprocess.run(['git','rev-parse','HEAD'],cwd=ROOT,text=True,capture_output=True); print('Git HEAD:',g.stdout.strip() or 'unavailable')


## 1–3. Repository Structure و Dataset Documentation

نقش مسیرها: `scripts/` اجرا و audit؛ `configs/` تنظیمات؛ `data/` ورودی local؛ `outputs/` شواهد measured؛ `reproduction/` قرارداد علمی؛ `notebooks/` artifact حاضر.

TED3 authoritative inputs از قبل extract شده‌اند و **نباید دوباره extract/download شوند**:
- `data/train/` labeled
- `data/test/` held-out test
- `data/TED3-unlabeled-data-15k-pseudo-mask/` unlabeled/pseudo-mask

Derived manifests فقط زیر `data/processed/ted3/`. pseudo-mask هرگز human ground truth فرض نمی‌شود.

In [ ]:
# هدف: tree محدود، inventory dataset، preflight و نمونه تصاویر.
# ورودی: repository paths و سه TED3 directory.
# خروجی: tree، count/size/extensions، sample gallery و preflight status.
# رفتار مورد انتظار: read-only؛ corpus بزرگ truncate می‌شود.
from collections import Counter
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from PIL import Image
for name in ['scripts','configs','data','outputs','reproduction','notebooks']:
 b=ROOT/name; print('\n##',name,b.exists())
 if b.exists():
  for i,p in enumerate(sorted(b.rglob('*'))):
   r=p.relative_to(b)
   if len(r.parts)<=2: print('  '*(len(r.parts)-1)+('📁 ' if p.is_dir() else '📄 ')+r.name)
   if i>120: print('... truncated ...'); break
D={'train':ROOT/'data/train','test':ROOT/'data/test','unlabeled':ROOT/'data/TED3-unlabeled-data-15k-pseudo-mask'}
EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}; rows=[]
for role,p in D.items():
 fs=[x for x in p.rglob('*') if x.is_file()] if p.is_dir() else []
 rows.append([role,str(p.relative_to(ROOT)),p.is_dir(),len(fs),round(sum(x.stat().st_size for x in fs)/1024**3,3),dict(Counter(x.suffix.lower() for x in fs).most_common(6))])
display(pd.DataFrame(rows,columns=['role','path','exists','files','GiB','extensions']))
pf=ROOT/'outputs/ted3_reproduction/preflight/input_directories.json'; preflight=json.loads(pf.read_text()) if pf.is_file() else {}
print('Preflight:',preflight.get('status','NOT_RUN'))
for role,p in D.items():
 imgs=[x for x in sorted(p.rglob('*')) if x.is_file() and x.suffix.lower() in EXT][:2] if p.is_dir() else []
 if imgs:
  fig,ax=plt.subplots(1,len(imgs),figsize=(10,4)); ax=np.atleast_1d(ax)
  for a,x in zip(ax,imgs): a.imshow(Image.open(x),cmap='gray'); a.set_title(str(x.relative_to(ROOT))); a.axis('off')
  plt.show()


In [ ]:
# هدف: exact duplicate leakage check با SHA256.
# ورودی: image files سه split.
# خروجی: تعداد hash مشترک.
# رفتار مورد انتظار: فقط با RUN_FULL_HASH_AUDIT=True اجرا می‌شود.
if RUN_FULL_HASH_AUDIT:
 def sha(p):
  h=hashlib.sha256()
  with p.open('rb') as f:
   for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
  return h.hexdigest()
 hs={r:{sha(x) for x in p.rglob('*') if x.is_file() and x.suffix.lower() in EXT} for r,p in D.items()}
 for a,b in [('train','test'),('train','unlabeled'),('test','unlabeled')]: print(a,b,len(hs[a]&hs[b]))
else: print('SHA256 audit skipped; RUN_FULL_HASH_AUDIT=False')


## 4–5. Paper-to-Code Mapping و Simulation Implementation

Publication contract: **32 classes، RGB 1024، 100 queries، 9 decoder layers، AdamW، LR=1e-4، steps 24000/25000، 26,250 iterations**. `burnin_iter=3000` در مخزن inferred است.

Reduced executable pipeline:
`PreparedTeeth → QuickSemiTransformer Teacher → CE+Dice → pseudo labels (low confidence ignored) → Student sup+unsup → EMA → collapse guard → evaluation → JSON/CSV/PNG`

TED3 research pipeline:
`preflight → forensic audit/preparation → method freeze → official checkpoint eval → supervised → semi-supervised → comparison → paper exports → delivery`.

In [ ]:
# هدف: بارگذاری قرارداد مقاله، mapping واقعی و stage matrix.
# ورودی: reference_contract، configs/paper.yaml، sourceهای موجود و outputs.
# خروجی: paper/config table، paper-to-code table و status همه فازهای TED3.
# رفتار مورد انتظار: reduced با paper architecture یکی فرض نمی‌شود؛ stage مفقود BLOCKED است.
contract=json.loads((ROOT/'reproduction/reference_contract.json').read_text())
try:
 import yaml; cfg=yaml.safe_load((ROOT/'configs/paper.yaml').read_text())
except: cfg={}
display(pd.DataFrame([{'parameter':k,'value':v} for k,v in cfg.items()]))
M=[
['Backbone/Encoder','QuickSemiTransformer.patch + TransformerEncoder','run_quick_real_experiment.py','Reduced surrogate; paper=MobileSAM/TinyViT'],
['Decoder/Queries','ConvTranspose decoder; reduced has no paper query mechanism','run_quick_real_experiment.py + configs/paper.yaml','Paper target=100 queries, 9 layers'],
['Losses','weighted CE + 0.5 Dice','supervised_loss()','Reduced measured'],
['Teacher/Student','warm-up → copy → pseudo supervision','main()','Reduced measured'],
['Pseudo labels','softmax/confidence; low-confidence=-100 ignore','main()','Not forced to background'],
['EMA/Distillation','teacher=.995T+.005S; sup+.05unsup','main()','Paper cfg ema=.9996; unsup=2.0'],
['Optimizer/Scheduler','AdamW; paper LR/steps in config','configs/paper.yaml','Reduced differs'],
['Metrics','IoU/Dice segmentation; P/R/F1 tooth-ID','batch_counts()/evaluate()','Paper evaluator frozen only at G1'],
['Upstream','pinned SemiT-SAM + deterministic patch','bootstrap_upstream.py; patch_upstream.py','vendor gitignored']]
display(pd.DataFrame(M,columns=['Paper Component','Implementation','Location','Note']))
ph=[
('0 preflight',['scripts/ted3_input_preflight.py'],['outputs/ted3_reproduction/preflight/input_directories.json']),
('1 dataset audit',['scripts/ted3_dataset_audit.py','scripts/audit_ted3_dataset.py'],['outputs/ted3_reproduction/dataset_audit']),
('2 method freeze',['scripts/freeze_ted3_method.py'],['reproduction/paper_method_manifest.json']),
('3 checkpoint eval',['scripts/evaluate_ted3_checkpoint.py','scripts/run_checkpoint_eval.py'],['outputs/ted3_reproduction/checkpoint_eval']),
('4 supervised',['scripts/run_ted3_supervised.py','scripts/train_ted3_supervised.py'],['outputs/ted3_reproduction/supervised','outputs/ted3_reproduction/compact_supervised']),
('5 semi-supervised',['scripts/run_ted3_semi_supervised.py','scripts/train_ted3_semi_supervised.py'],['outputs/ted3_reproduction/semi_supervised','outputs/ted3_reproduction/compact_semi_supervised']),
('6 comparison',['scripts/compare_ted3_experiments.py'],['outputs/ted3_reproduction/comparison']),
('7 exports',['scripts/export_paper_style_figures.py'],['outputs/ted3_reproduction/paper_exports','outputs/paper_style']),
('8 delivery',['scripts/package_client_delivery.py','scripts/package_deliverable.py'],['outputs/ted3_reproduction/delivery'])]
rows=[]
for n,s,o in ph:
 sf=[x for x in s if (ROOT/x).is_file()]; of=[x for x in o if (ROOT/x).exists()]
 rows.append([n,sf,of,'ARTIFACT_PRESENT' if of else ('ENTRYPOINT_PRESENT' if sf else 'BLOCKED')])
display(pd.DataFrame(rows,columns=['phase','entrypoints','artifacts','status']))


## 6–7. Safe Execution, Existing Results, Paper Comparison و Visualization

Safe commands:
`python project.py ted3-preflight`؛ `python project.py audit`؛ `python scripts/export_paper_style_figures.py`

Reduced run:
`python scripts/run_quick_real_experiment.py` سپس `python scripts/validate_final_outputs.py`.

`project.py full` historical reduced track است، **نه** TED3 full-scale.

In [ ]:
# هدف: runner امن، load metrics، Published/Measured/Gap و figures.
# ورودی: flagها و outputs موجود.
# خروجی: command status، metrics catalog، comparison و gallery.
# رفتار مورد انتظار: full training خودکار نیست و metric مفقود fabricate نمی‌شود.
STAGES={'preflight':([sys.executable,'project.py','ted3-preflight'],True),'audit':([sys.executable,'project.py','audit'],True),
'reduced':([sys.executable,'scripts/run_quick_real_experiment.py'],False),'figures':([sys.executable,'scripts/export_paper_style_figures.py'],True)}
def run_stage(n):
 cmd,safe=STAGES[n]
 if safe and not RUN_SAFE_COMMANDS:return print('[SKIP]',n)
 if n=='reduced' and not RUN_REDUCED_SIMULATION:return print('[SKIP] reduced')
 p=subprocess.run(cmd,cwd=ROOT,text=True,capture_output=True); print(' '.join(map(shlex.quote,cmd)),p.returncode); print(p.stdout[-3000:]); print(p.stderr[-1500:])
if RUN_SAFE_COMMANDS: run_stage('preflight'); run_stage('audit')
if RUN_REDUCED_SIMULATION: run_stage('reduced')
MET=['iou','dice','precision','recall','f1']; rec=[]
for base in [ROOT/'outputs/ted3_reproduction',ROOT/'outputs/paper_reproduction',ROOT/'outputs/final']:
 if base.exists():
  for p in base.rglob('*.json'):
   try:o=json.loads(p.read_text())
   except:continue
   if isinstance(o,dict) and all(isinstance(o.get(k),(int,float)) for k in MET): rec.append({'source':str(p.relative_to(ROOT)),**{k:float(o[k]) for k in MET}})
display(pd.DataFrame(rec))
prio=['semi_supervised','compact_semi_supervised','supervised','compact_supervised','checkpoint_eval','outputs/final']
sel=next((r for t in prio for r in rec if t in r['source']),None); pub=contract['publication']['published_metrics_percent']
display(pd.DataFrame([[k.upper(),pub[k],None if not sel else sel[k],None if not sel else sel[k]-pub[k],None if not sel else sel['source']] for k in MET],
 columns=['Metric','Published SemiTNet','Measured Result','Gap','Measured Source']))
figs=[p for d in [ROOT/'outputs/ted3_reproduction/compact_supervised/figures',ROOT/'outputs/ted3_reproduction/compact_semi_supervised/figures',ROOT/'outputs/ted3_reproduction/supervised/figures',ROOT/'outputs/ted3_reproduction/semi_supervised/figures',ROOT/'outputs/final/figures',ROOT/'outputs/paper_style/figures'] if d.is_dir() for p in d.rglob('*.png')]
for p in figs[:20]: plt.figure(figsize=(10,5)); plt.imshow(Image.open(p)); plt.title(str(p.relative_to(ROOT))); plt.axis('off'); plt.show()


## 8. Reproducibility Gates، Checklist و Final Summary

- **G0:** reduced baseline integrity.
- **G1:** official checkpoint روی exact paper test/evaluator.
- **G2:** architecture/data/preprocessing/loss/schedule/evaluator freeze.
- **G3:** full paper-faithful training.

Known limitations: reduced run paper-equivalent نیست؛ TED3 بدون identity evidence = exact TSI15k نیست؛ vendor upstream gitignored است؛ full 26,250 iterations اینجا اجرا نشده است.

عبارت مجاز برای historical reduced baseline:

> **Measured reduced-scale re-simulation; not numerically equivalent to the published full-scale experiment.**

In [ ]:
# هدف: نمایش audit G0-G3 و checklist نهایی.
# ورودی: reproducibility_audit.json و artifacts.
# خروجی: gate table و checklist.
# رفتار مورد انتظار: G0 PASS هرگز paper reproduction تفسیر نمی‌شود.
ap=ROOT/'outputs/reproducibility_audit.json'; audit=json.loads(ap.read_text()) if ap.is_file() else {}
if audit:
 display(pd.DataFrame([[g,audit.get(g,{}).get('status'),audit.get(g,{}).get('reason')] for g in ['G0_reduced_baseline','G1_official_checkpoint','G2_method_faithfulness','G3_training_reproduction']],columns=['gate','status','reason']))
 print('Overall:',audit.get('overall_classification')); print(audit.get('defensible_statement'))
check=[
['Dataset paths',all(p.is_dir() for p in D.values())],['Preflight',bool(preflight)],
['Training config',(ROOT/'configs/paper.yaml').is_file()],['Evaluator gate',(ROOT/'scripts/reproducibility_gate.py').is_file()],
['Figures',bool(figs)],['Measured metrics',sel is not None]]
display(pd.DataFrame(check,columns=['Checklist item','PASS']))
